# Autonomous Driving with Deep RL — Training, Evaluation, Visualization

**Giuseppe D'Auria** — Reinforcement Learning 2025/2026, University of Padova
GitHub: https://github.com/cocopops9/RL-based_autonomous_driving

A single notebook that trains a Double + Dueling DQN agent on `highway-v0`,
evaluates it honestly, and shows the car driving as an inline animation.

**Sections**
1. Setup and Google Drive
2. Install the project code
3. Dependencies
4. Heuristic baseline
5. Training (resumable, checkpoints every 5 min)
6. Evaluation on held-out test seeds (100 episodes)
7. Visualization — inline animation of baseline vs agent
8. Training plots
9. Download

A Colab runtime has no desktop, so a native pop-up window is not possible.
Section 7 records frames in `rgb_array` mode and plays them back inline, which
is the notebook equivalent of the graphical window and is what the report
screenshots are taken from. To open a real OS window, run `visualize.py` on a
local machine (command shown in Section 7).

## 1. Setup and Google Drive
Checkpoints and results live on Drive so they survive a disconnect.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount failed:", e)
    print("This is a Google auth error, not a code bug. Fixes:")
    print(" - complete the whole authorization popup (Allow on every screen)")
    print(" - allow third-party cookies for google.com / googleusercontent.com")
    print(" - Runtime > Disconnect and delete runtime, then run this cell first")
    raise

import os
WORK_DIR = '/content/drive/MyDrive/rl_project_ad'
os.makedirs(WORK_DIR, exist_ok=True)
print("Working directory:", WORK_DIR)

## 2. Install the project code
Upload the project zip once. Only code files are copied; `weights/` and `results/` on Drive are preserved.

In [ ]:
FORCE_UPDATE = False
import os, shutil, zipfile

CODE_FILES = ["dqn_agent.py", "training.py", "evaluate.py", "your_baseline.py",
              "manual_control.py", "plot_results.py", "visualize.py",
              "safety_filter.py", "README.md"]

need = FORCE_UPDATE or not os.path.exists(f"{WORK_DIR}/safety_filter.py")
if need:
    from google.colab import files
    print("Upload the project zip now...")
    up = files.upload()
    zip_name = list(up.keys())[0]
    ex = "/tmp/proj"; shutil.rmtree(ex, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(ex)
    base = None
    for root, _, fs in os.walk(ex):
        if "dqn_agent.py" in fs:
            base = root; break
    if base is None:
        raise FileNotFoundError("dqn_agent.py not found in the uploaded zip")
    os.makedirs(f"{WORK_DIR}/weights", exist_ok=True)
    os.makedirs(f"{WORK_DIR}/results", exist_ok=True)
    copied = []
    for fn in CODE_FILES:
        src = os.path.join(base, fn)
        if os.path.exists(src):
            shutil.copy(src, f"{WORK_DIR}/{fn}"); copied.append(fn)
    shutil.rmtree(ex, ignore_errors=True)
    print("Installed:", copied)
else:
    print("Code already on Drive (set FORCE_UPDATE=True to overwrite).")

os.chdir(WORK_DIR)
print("Contents:", sorted(os.listdir(".")))

## 3. Dependencies
Re-run after every reconnect; Colab wipes installed packages.

In [ ]:
!pip install gymnasium highway-env imageio imageio-ffmpeg -q
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Heuristic baseline
A defensive rule-based policy; the benchmark the agent must beat. Skipped if already computed.

In [ ]:
import os, json
if not os.path.exists("results/baseline_results.json"):
    !python your_baseline.py
else:
    r = json.load(open("results/baseline_results.json"))
    print(f"baseline: mean={r['mean_return']:.2f}  crash={r['crash_rate']:.0%}")

## 5. Training (resumable)
Runs 50k environment steps. Full-state atomic checkpoints every 5 min let it
resume after a disconnect: just re-run this cell. If training already finished,
it detects the checkpoint and skips straight to the final evaluation.

In [ ]:
!python training.py

## 6. Evaluation on held-out test seeds
100 episodes on seeds `90000+i`, disjoint from the selection seeds used during
training. The crash rate is printed with its binomial standard error; a
10-episode run can read 0% purely by luck, so we use 100.

In [ ]:
# 6a. Raw network
!python evaluate.py --episodes 100

In [ ]:
# 6b. Strict safety filter
!python evaluate.py --episodes 100 --strict

In [ ]:
# 6c. Conservative safety filter (lowest crash rate)
!python evaluate.py --episodes 100 --conservative

## 7. Visualization — inline animation

Records episodes in `rgb_array` mode and plays them back inline as a video.
The ego vehicle is green. Pick the policy with `POLICY` below:
`baseline`, `agent`, `agent_strict`, or `agent_conservative`.

For a real pop-up OS window, run locally instead:
```
python visualize.py both --seed 90000
python visualize.py agent --conservative --seed 90000
```

In [ ]:
# --- Inline animation: several consecutive episodes joined into one longer clip ---
import numpy as np, gymnasium, highway_env, imageio, os
from IPython.display import Video, display

# ---- choose what to watch ----
POLICY   = "agent_conservative"   # baseline | agent | agent_strict | agent_conservative
SEED     = 90003                  # first episode seed; later episodes use SEED+1, +2, ...
EPISODES = 6                      # how many episodes to string together (longer video)
FPS      = 15                     # playback smoothness
# ------------------------------

CFG = {"action": {"type": "DiscreteMetaAction"},
       "lanes_count": 3, "ego_spacing": 1.5,
       "vehicles_count": 50, "duration": 40}

def make_policy(name):
    if name == "baseline":
        from your_baseline import heuristic_action
        return ("raw", (lambda obs2d: heuristic_action(obs2d)), None)
    from dqn_agent import DQNAgent
    agent = DQNAgent(state_dim=25, action_dim=5, hidden_dim=128)
    agent.load("weights/dqn_highway.pt")
    filt = None
    if name == "agent_strict":
        from safety_filter import StrictFilter; filt = StrictFilter()
    elif name == "agent_conservative":
        from safety_filter import ConservativeFilter; filt = ConservativeFilter()
    return ("net", agent, filt)

kind, policy, filt = make_policy(POLICY)
env = gymnasium.make("highway-v0", config=CFG, render_mode="rgb_array")

all_frames, returns, crashes = [], [], []
for k in range(EPISODES):
    obs, _ = env.reset(seed=SEED + k)
    if filt is not None: filt.reset()
    done = trunc = False; ret = 0.0
    while not (done or trunc):
        if kind == "raw":
            action = policy(obs)
        else:
            flat = obs.reshape(-1)
            action = policy.select_action(flat, evaluate=True)
            if filt is not None: action = filt.filter(action, flat)
        obs, r, done, trunc, _ = env.step(action)
        all_frames.append(env.render()); ret += r
    returns.append(ret); crashes.append(done)
    # short pause (repeat last frame) between episodes as a visual separator
    for _ in range(FPS // 2):
        all_frames.append(all_frames[-1])
env.close()

out = f"/content/{POLICY}_x{EPISODES}.mp4"
imageio.mimsave(out, all_frames, fps=FPS)
print(f"{POLICY}: {EPISODES} episodes, seeds {SEED}..{SEED+EPISODES-1}")
print(f"  returns: " + ", ".join(f"{x:.1f}" for x in returns))
print(f"  crashes: {sum(crashes)}/{EPISODES}   total frames: {len(all_frames)} "
      f"(~{len(all_frames)/FPS:.0f}s at {FPS} fps)")
display(Video(out, embed=True, width=680))

### Side-by-side: baseline vs agent on the same traffic
Runs both policies on one seed and shows the two clips, for the report figure.

In [ ]:
# --- Side-by-side longer clips: baseline vs agent over the same episodes ---
import numpy as np, gymnasium, highway_env, imageio
from IPython.display import Video, display

CFG = {"action": {"type": "DiscreteMetaAction"},
       "lanes_count": 3, "ego_spacing": 1.5,
       "vehicles_count": 50, "duration": 40}
SEED     = 90003
EPISODES = 5
FPS      = 15

def rollout_many(name):
    kind, policy, filt = make_policy(name)
    env = gymnasium.make("highway-v0", config=CFG, render_mode="rgb_array")
    frames, rets, crs = [], [], []
    for k in range(EPISODES):
        obs, _ = env.reset(seed=SEED + k)
        if filt is not None: filt.reset()
        done = trunc = False; ret = 0.0
        while not (done or trunc):
            if kind == "raw":
                action = policy(obs)
            else:
                flat = obs.reshape(-1)
                action = policy.select_action(flat, evaluate=True)
                if filt is not None: action = filt.filter(action, flat)
            obs, r, done, trunc, _ = env.step(action)
            frames.append(env.render()); ret += r
        rets.append(ret); crs.append(done)
        for _ in range(FPS // 2):
            frames.append(frames[-1])
    env.close()
    path = f"/content/cmp_{name}_x{EPISODES}.mp4"
    imageio.mimsave(path, frames, fps=FPS)
    return path, rets, crs

for name in ["baseline", "agent_conservative"]:
    p, rets, crs = rollout_many(name)
    print(f"{name:20s} returns=" + ",".join(f"{x:.1f}" for x in rets) +
          f"  crashes={sum(crs)}/{EPISODES}")
    display(Video(p, embed=True, width=620))

### Save report stills
Writes `shot_baseline.png` and `shot_agent.png` into `report/results/` for the LaTeX figure.

In [ ]:
# --- Save representative stills for the report (report/results/) ---
# Captures one clean mid-episode frame of the baseline and of the agent on the
# same seed, writes them where the LaTeX report expects them.
import os, numpy as np, gymnasium, highway_env, imageio

os.makedirs("report/results", exist_ok=True)
CFG = {"action": {"type": "DiscreteMetaAction"},
       "lanes_count": 3, "ego_spacing": 1.5,
       "vehicles_count": 50, "duration": 40}
SHOT_SEED = 90003          # a seed where the agent drives cleanly

def grab_still(name, out_png, step_at=20):
    kind, policy, filt = make_policy(name)
    if filt is not None: filt.reset()
    env = gymnasium.make("highway-v0", config=CFG, render_mode="rgb_array")
    obs, _ = env.reset(seed=SHOT_SEED)
    frame = env.render()
    done = trunc = False; step = 0
    while not (done or trunc):
        if kind == "raw":
            action = policy(obs)
        else:
            flat = obs.reshape(-1)
            action = policy.select_action(flat, evaluate=True)
            if filt is not None: action = filt.filter(action, flat)
        obs, _, done, trunc, _ = env.step(action)
        frame = env.render(); step += 1
        if step >= step_at:      # grab a representative mid-episode frame
            break
    env.close()
    imageio.imwrite(out_png, frame)
    print("saved", out_png, frame.shape)

grab_still("baseline",           "report/results/shot_baseline.png")
grab_still("agent_conservative", "report/results/shot_agent.png")

from IPython.display import Image, display
display(Image("report/results/shot_baseline.png"))
display(Image("report/results/shot_agent.png"))

## 8. Training plots

In [ ]:
!python plot_results.py
from IPython.display import Image, display
import os
for img in ["results/training_curve.png", "results/eval_returns.png",
            "results/crash_rate.png", "results/loss_curve.png"]:
    if os.path.exists(img):
        display(Image(img))

## 9. Download everything

In [ ]:
import shutil
shutil.make_archive('/content/rl_project_ad_trained', 'zip', WORK_DIR)
from google.colab import files
files.download('/content/rl_project_ad_trained.zip')

## Recovery after a disconnect
Reconnect, then Runtime > Run all. Sections 2 and 4 skip automatically, and
Section 5 resumes training from the last checkpoint. To restart from scratch:
`!python training.py --no-resume`.